<a href="https://colab.research.google.com/github/VikaSvyat/DI_Bootcamp/blob/main/Week14/BERT_Trustworthy_W14D1_DC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
####without output for GitHub

# Week 14. Daily Challenge : Building Trustworthy Insights with BERT

https://octopus.developers.institute/courses/collection/124/course/725/section/1980/chapter/4742

## 0. Importing Libraries

In [ ]:
import datasets
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, TrainingArguments, Trainer
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score

## 1. Data Loading & Inspection

Load the tweet_eval dataset (sentiment configuration) via datasets.load_dataset.



In [ ]:
dataset = datasets.load_dataset("cardiffnlp/tweet_eval", "sentiment")

In [ ]:
dataset

Print dataset splits and class distribution. Confirm there are 3 labels (negative, neutral, positive).

In [ ]:
def print_split_distrib(split):
  print(f'--- Dataset split : {split} ---')
  class_distrib = [0, 0, 0]

  for i in range (0, dataset[split].num_rows):
    i_label = dataset[split][i]['label']
    class_distrib[i_label] += 1

  print(class_distrib)

  for number in range(0, 3):
    print(f'Label {number} : Distribution = {class_distrib[number]/dataset[split].num_rows*100}')

In [ ]:
print_split_distrib('train')
print_split_distrib('test')
print_split_distrib('validation')

Save two example tweets per label for later visualization.

In [ ]:
from collections import defaultdict

examples = defaultdict(list)
target_labels = {0, 1, 2}

for item in dataset["train"]:
    label = item["label"]
    text = item["text"]

    if len(examples[label]) < 2:
        examples[label].append(text)

    # select 2 examples for each label
    if all(len(examples[l]) == 2 for l in target_labels):
        break

label_map = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

for label, texts in examples.items():
    print(f"{label_map[label]}:")
    for t in texts:
        print("-", t)

## 2. Tokenization Pipeline

Initialize AutoTokenizer with distilbert-base-uncased.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

Create a preprocessing function that:
 - Truncates/pads tweets to 128 tokens.
- Returns input_ids, attention_mask, and labels.

In [ ]:
def preprocess(batch):
  enc = tokenizer(batch['text'], max_length=128, padding='max_length', truncation=True)
  enc['labels'] = batch['label']
  return enc

dataset_enc = dataset.map(preprocess, batched=True)

dataset_enc

### TODO : Shuffle the items

Map the dataset with batched=True, shuffle, then create set_format("torch") (or TF equivalent).

In [ ]:
dataset_enc = dataset_enc.shuffle(seed=42)

In [ ]:
dataset_enc.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

dataset_enc

## 3. Fine-Tuning Setup

Load AutoModelForSequenceClassification with 3 labels.

Use TrainingArguments (epochs=3, batch_size=32, lr=5e-5, weight_decay=0.01).

In [ ]:
# Instanciating our model (3 labels)
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3)

training_args = TrainingArguments(num_train_epochs=3,
                                  per_device_train_batch_size=32,
                                  learning_rate=5e-5,
                                  weight_decay=0.01,
                                  load_best_model_at_end=True,
                                  eval_strategy='epoch',
                                  save_strategy='epoch')

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"CUDA device name: {torch.cuda.get_device_name(0)}")

Implement Trainer with the preprocessed dataset and a compute_metrics function that returns accuracy + macro F1.


In [ ]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)
  accuracy = accuracy_score(y_true=labels, y_pred=predictions)
  macro_f1 = f1_score(y_true=labels, y_pred=predictions, average='macro')
  return {'accuracy': accuracy, 'macro_f1': macro_f1}


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_enc['train'],
    eval_dataset=dataset_enc['test'],
    compute_metrics=compute_metrics
)



Train and save the best checkpoint (use load_best_model_at_end=True).

In [ ]:
trainer_results = trainer.train()

## 4. Evaluation & Calibration

- Evaluate on the validation split; log accuracy and F1.

- Collect the softmax scores for predicted classes on the test split.

- Plot a histogram of confidence scores (bins of 0.1).



In [ ]:
val_metrics = trainer.evaluate(dataset_enc['validation'])
val_metrics

In [ ]:
test_predictions = trainer.predict(dataset_enc['test'])
logits = torch.tensor(test_predictions.predictions)
labels = test_predictions.label_ids
probs = torch.softmax(logits, dim=-1).numpy()
max_confidence = probs.max(axis=1)

In [ ]:
# Plotting the confidence
plt.figure(figsize=(10, 6))
plt.hist(max_confidence, bins=10, range=(0, 1), edgecolor='blue')
plt.xlabel('Confidence')
plt.ylabel('Frequency')
plt.title('Confidence Distribution')
plt.show

- Comment on over/under-confidence trends.

The confidence distribution shows that most predictions fall between 0.6 and 1.0, indicating that the model is generally confident. However, there is also a noticeable number of predictions around 0.5, suggesting uncertainty for some inputs. The model rarely produces very low confidence scores, meaning it tends to commit to a class rather than expressing strong doubt. Overall, the model is moderately confident but not extremely overconfident.

## 5. Attention Inspection

In [ ]:
example_tweet = dataset['test'][3]['text']
example_tweet = "I feel terrible, it's so bad, I am sick, and I have a dog!"
example_tweet

In [ ]:
base_model = AutoModel.from_pretrained("distilbert-base-uncased", output_attentions=True)

In [ ]:
base_model_2 = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", output_attentions=True)

In [ ]:
# Tokenize our example tweet
example_tweet_enc = tokenizer.encode(example_tweet, return_tensors='pt')
example_tweet_enc

In [ ]:
outputs_1 = base_model(example_tweet_enc)
outputs_2 = base_model_2(example_tweet_enc)

In [ ]:
# outputs_1 is from AutoModel, which does not provide classification logits.
# To get a prediction, we need to use outputs_2 from AutoModelForSequenceClassification.

# Extract logits from outputs_2
logits = outputs_2.logits

# Apply softmax to get probabilities (optional, but good for understanding confidence)
probabilities = torch.softmax(logits, dim=-1)

# Get the predicted label using argmax
predicted_label = torch.argmax(logits, dim=-1).item()

print(f"Raw logits: {logits.tolist()}")
print(f"Probabilities: {probabilities.tolist()}")
print(f"Predicted label: {predicted_label}")

In [ ]:
attn_map = outputs_2.attentions[-1]
attn = attn_map

cls_attention = attn[0, :, 0, :].mean(dim=0).detach().numpy()
tokens = tokenizer.convert_ids_to_tokens(example_tweet_enc[0])

cls_attention

In [ ]:
plt.figure(figsize=(12,2))
plt.bar(range(len(tokens)), cls_attention, color="#FF6B6B")
plt.xticks(ticks=range(len(tokens)), labels=tokens, rotation=90)
plt.ylabel("Attention from [CLS]")
plt.title("Attention Focus on Example Tweet (without Seaborn)")
plt.tight_layout()
plt.savefig("attention_example_no_seaborn.png", dpi=200)
plt.show()

The model assigns higher attention to structural tokens like "I" and "and", while sentiment-heavy words like "terrible" receive lower attention from the [CLS] token. This suggests that attention does not directly correspond to word importance. Instead, semantic information from sentiment words may already be integrated in earlier layers or captured by specific attention heads, which gets diluted when averaging across heads.